### 长期记忆

#### 基础 api 使用

In [1]:
import os
from typing import NotRequired
from venv import create

from langchain.agents import create_agent, AgentState
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.store.postgres import PostgresStore
from dotenv import load_dotenv
from pydantic_ai import agent

load_dotenv(override=True)

store = InMemoryStore()
namespace = ("users",)
user_id = "user-1"
user_name = "小许"

store.put(namespace, user_id, {"name": user_name})

item = store.get(namespace, user_id)
print(item)

# 更新数据
user_name = "小红"
store.put(namespace, user_id, {"name": user_name})

item = store.get(namespace, user_id)
print(item)

/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Item(namespace=['users'], key='user-1', value={'name': '小许'}, created_at='2026-09-24T07:36:28.226778+00:00', updated_at='2026-09-24T07:36:28.226780+00:00')
Item(namespace=['users'], key='user-1', value={'name': '小红'}, created_at='2026-09-24T07:36:28.226921+00:00', updated_at='2026-09-24T07:36:28.226921+00:00')


#### 基于 postgreSQL

In [2]:

with PostgresStore.from_conn_string(os.getenv("DATABASE_URL")) as store1:
    store1.setup()
    namespace = ("users",)
    user_id = "user-1"
    user_name = "小许"

    store1.put(namespace, user_id, {"name": user_name})

    item = store1.get(namespace, user_id)
    print(item)

Item(namespace=['users'], key='user-1', value={'name': '小许'}, created_at='2026-09-24T14:26:47.454020+08:00', updated_at='2026-09-24T15:36:28.253720+08:00')


In [3]:
### search使用
for item in store.search(("users",)):
    print(item)

Item(namespace=['users'], key='user-1', value={'name': '小红'}, created_at='2026-09-24T07:36:28.226921+00:00', updated_at='2026-09-24T07:36:28.226921+00:00', score=None)


In [4]:
# filter 搜索：按 value 中的字段做精确过滤
store.put(("users",), "user-2", {"name": "小许", "role": "admin"})
store.put(("users",), "user-3", {"name": "小许", "role": "guest"})
store.put(("users",), "user-4", {"name": "小红", "role": "admin"})

print("name == 小许：")
for item in store.search(("users",), filter={"name": "小许"}):
    print(" ", item.key, item.value)

print("role == admin：")
for item in store.search(("users",), filter={"role": "admin"}):
    print(" ", item.key, item.value)


name == 小许：
  user-2 {'name': '小许', 'role': 'admin'}
  user-3 {'name': '小许', 'role': 'guest'}
role == admin：
  user-2 {'name': '小许', 'role': 'admin'}
  user-4 {'name': '小红', 'role': 'admin'}


#### namespace 是层级路径

`namespace` 是一个字符串元组，作用类似「文件夹路径」，用来做多级分组与隔离。

| 概念 | 类型 | 类比 |
| --- | --- | --- |
| `namespace` | `tuple[str, ...]` | 文件夹路径 |
| `key` | `str` | 文件名 |
| `value` | `dict` | 文件内容 |

约定：通常第一段是「类别」，第二段是「归属 id（用户/助手/会话）」，后面是子分类，例如 `("users", user_id, "profile")`、`("memories", user_id)`。

- `get` / `put` / `delete` 必须传入**完整** namespace + key；
- `search(namespace_prefix)` 是**前缀匹配**，会返回以该前缀开头的所有条目（含更深层级）。

In [5]:
# namespace 是元组，表示层级路径，可实现多级隔离
store.put(("users", "user-1", "profile"), "basic", {"name": "小许"})
store.put(("users", "user-1", "memories"), "m1", {"text": "喜欢 Python"})
store.put(("users", "user-2", "profile"), "basic", {"name": "小红"})

print("search(('users', 'user-1')) —— 只命中 user-1 下的条目：")
for item in store.search(("users", "user-1")):
    print(" ", item.namespace, item.key, item.value)

print("\nsearch(('users',)) —— 前缀匹配，命中所有用户：")
for item in store.search(("users",)):
    print(" ", item.namespace, item.key)

print("\nlist_namespaces(prefix=('users',))：")
for ns in store.list_namespaces(prefix=("users",)):
    print(" ", ns)


search(('users', 'user-1')) —— 只命中 user-1 下的条目：
  ('users', 'user-1', 'profile') basic {'name': '小许'}
  ('users', 'user-1', 'memories') m1 {'text': '喜欢 Python'}

search(('users',)) —— 前缀匹配，命中所有用户：
  ('users',) user-1
  ('users',) user-2
  ('users',) user-3
  ('users',) user-4
  ('users', 'user-1', 'profile') basic
  ('users', 'user-1', 'memories') m1
  ('users', 'user-2', 'profile') basic

list_namespaces(prefix=('users',))：
  ('users',)
  ('users', 'user-1', 'memories')
  ('users', 'user-1', 'profile')
  ('users', 'user-2', 'profile')


#### 删除与列举

In [6]:
# 删除单条记忆
store.delete(("users",), "user-4")
print("删除 user-4 后 users 下的 key：", [item.key for item in store.search(("users",))])

# 只看顶层命名空间（max_depth 限制层级深度）
print("max_depth=2：", store.list_namespaces(max_depth=2))


删除 user-4 后 users 下的 key： ['user-1', 'user-2', 'user-3', 'basic', 'm1', 'basic']
max_depth=2： [('users',), ('users', 'user-1'), ('users', 'user-2')]


#### PostgreSQL：search 与前缀 namespace

`PostgresStore` 的 API 与 `InMemoryStore` 完全一致，只是把数据落到 PostgreSQL。（异步场景使用 `AsyncPostgresStore`，其 `from_conn_string` 是异步上下文管理器。）

In [7]:
with PostgresStore.from_conn_string(os.getenv("DATABASE_URL")) as pg:
    pg.setup()

    ns = ("demo", "users")
    pg.put(ns, "user-1", {"name": "小许", "role": "admin"})
    pg.put(ns, "user-2", {"name": "小红", "role": "guest"})

    print("get：", pg.get(ns, "user-1").value)

    print("\nsearch(('demo', 'users'))：")
    for item in pg.search(ns):
        print(" ", item.key, item.value)

    print("\nfilter role == admin：")
    for item in pg.search(ns, filter={"role": "admin"}):
        print(" ", item.key, item.value)

    print("\nlist_namespaces(prefix=('demo',))：", pg.list_namespaces(prefix=("demo",)))


get： {'name': '小许', 'role': 'admin'}

search(('demo', 'users'))：
  user-2 {'name': '小红', 'role': 'guest'}
  user-1 {'name': '小许', 'role': 'admin'}

filter role == admin：
  user-1 {'name': '小许', 'role': 'admin'}

list_namespaces(prefix=('demo',))： [('demo', 'users')]


### 基于嵌入的语义检索（Embedding）

前面 `search` 只能按 namespace 前缀和 `filter` 精确过滤。要用**自然语言查询**（语义相似度）检索，
需要给 Store 配置**嵌入索引**：

```python
store = InMemoryStore(
    index={"dims": 向量维度, "embed": 嵌入函数或模型, "fields": 要索引的字段}
)
```

- `dims`：向量维度，必须与嵌入模型的输出维度一致；
- `embed`：嵌入函数，或任意 `Embeddings` 对象（如 `OpenAIEmbeddings`）；
- `fields`：要嵌入的字段，默认 `["$"]`（整个 value）；也可写 `["text"]`，支持嵌套如 `"metadata.title"`；
- `put(ns, key, value, index=[...])` 可在写入时覆盖该条要索引的字段；
- `search(ns, query="...")` 会按相似度排序，结果带 `score` 字段。

In [8]:
import hashlib

from langchain_core.embeddings import Embeddings

DIMS = 128


class LocalHashEmbeddings(Embeddings):
    """离线占位嵌入：用字符 bigram 哈希成固定维度向量。

    仅用于本地演示 Store 的语义检索 API，不依赖任何外部服务；
    它只体现「字面相似度」，生产环境请替换为真实嵌入模型（见下一节）。
    """

    def _vec(self, text: str) -> list[float]:
        vec = [0.0] * DIMS
        text = text.lower()
        for i in range(len(text) - 1):
            gram = text[i: i + 2]
            vec[int(hashlib.md5(gram.encode()).hexdigest(), 16) % DIMS] += 1.0
        norm = sum(v * v for v in vec) ** 0.5 or 1.0
        return [v / norm for v in vec]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._vec(t) for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._vec(text)


In [9]:
# 给 Store 配置嵌入索引后，search 就能用自然语言 query 做相似度检索
mem_store = InMemoryStore(
    index={"dims": DIMS, "embed": LocalHashEmbeddings(), "fields": ["text"]}
)

ns = ("memories", "user-1")
mem_store.put(ns, "m1", {"text": "我喜欢用 Python 写后端"})
mem_store.put(ns, "m2", {"text": "我养了一只叫豆豆的猫"})
mem_store.put(ns, "m3", {"text": "我在学习 LangGraph 工作流编排"})

for query in ["Python 后端开发", "LangGraph 工作流"]:
    print("query:", query)
    for item in mem_store.search(ns, query=query, limit=3):
        print(f"  {item.key}  score={item.score:.3f}  {item.value['text']}")


query: Python 后端开发
  m1  score=0.650  我喜欢用 Python 写后端
  m2  score=0.192  我养了一只叫豆豆的猫
  m3  score=0.066  我在学习 LangGraph 工作流编排
query: LangGraph 工作流
  m3  score=0.795  我在学习 LangGraph 工作流编排
  m2  score=0.096  我养了一只叫豆豆的猫
  m1  score=0.000  我喜欢用 Python 写后端


#### 说明与替换为真实嵌入模型

上面用的 `LocalHashEmbeddings` 只是**离线占位实现**（字符 bigram 哈希），
目的是在没有任何外部服务的情况下演示 Store 的语义检索 API，它只反映字面相似度。

生产中请替换为真实嵌入模型（`OpenAIEmbeddings` / `OllamaEmbeddings` 等），
并保证 `dims` 与模型输出维度一致。示例见下一个单元。

注意：`PostgresStore` 的语义检索依赖数据库的 **pgvector** 扩展；
若数据库未安装，`query` 检索会报 `extension "vector" is not available`。

In [10]:
# ===== 换成真实的嵌入模型 =====
# from langchain_openai import OpenAIEmbeddings
# from langchain_ollama import OllamaEmbeddings
#
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")   # 需 OPENAI_API_KEY
# # embeddings = OllamaEmbeddings(model="nomic-embed-text")       # 需本地 Ollama
# # 注意：dims 必须和嵌入模型的输出维度一致（text-embedding-3-small 为 1536）
# store = InMemoryStore(index={"dims": 1536, "embed": embeddings, "fields": ["text"]})

# ===== PostgreSQL 语义检索 =====
# 需要数据库安装 pgvector 扩展（CREATE EXTENSION vector），否则 query 检索会失败
try:
    with PostgresStore.from_conn_string(
            os.getenv("DATABASE_URL"),
            index={"dims": DIMS, "embed": LocalHashEmbeddings(), "fields": ["text"]},
    ) as pg:
        pg.setup()

        ns = ("demo_mem", "user-1")
        pg.put(ns, "m1", {"text": "我喜欢用 Python 写后端"})
        pg.put(ns, "m2", {"text": "我养了一只叫豆豆的猫"})

        print("PostgreSQL 语义检索：")
        for item in pg.search(ns, query="Python 后端开发", limit=3):
            print(f"  {item.key}  score={item.score:.3f}  {item.value['text']}")
except Exception as exc:
    print("PostgreSQL 语义检索不可用：", type(exc).__name__, str(exc)[:60])
    print("请先在数据库启用 pgvector：CREATE EXTENSION vector;（或使用 pgvector/pgvector 镜像）")


PostgreSQL 语义检索不可用： FeatureNotSupported extension "vector" is not available
DETAIL:  Could not open 
请先在数据库启用 pgvector：CREATE EXTENSION vector;（或使用 pgvector/pgvector 镜像）


In [11]:


@tool
def save_user_info(name: str, job: str, runtime: ToolRuntime) -> str:
    """
    保存用户信息
    :param job: 用户职业
    :param name: 用户姓名
    :param runtime: 工具运行时上下文
    :return:
    """
    user_id = runtime.state.get("user_id")
    namespace = ("users",)
    runtime.store.put(namespace, user_id, {"name": name, "job": job})
    return "saved user"


@tool
def get_user_info(runtime: ToolRuntime) -> str:
    """
    获取用户信息
    :param runtime: 工具运行时上下文
    :return: 用户信息
    """
    user_id = runtime.state.get("user_id")
    item = runtime.store.get(namespace, user_id)
    return str(item.value) if item else "unknown"


class CustomState(AgentState):
    user_id: NotRequired[str]


store = InMemoryStore()
agent_long = create_agent(
    model="deepseek:deepseek-flash",
    tools=[get_user_info, save_user_info],
    store=store,
    state_schema=CustomState,
    system_prompt="你是一个AI助手，遇到用户提问存在个人信息要使用工具保存，如果户询问个人信息，要使用工具查询。使用杀生鱼丸的口吻回答。"
)

response1 = agent_long.invoke(
    {"messages": [HumanMessage("你好啊，我叫小雪是一个Java后端开发工程师")], "user_id": "user-1"})
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent_long.invoke(
    {"messages": [HumanMessage("你还记得我吗？")], "user_id": "user-1"})
for msg in response2["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊，我叫小雪是一个Java后端开发工程师
================================== Ai Message ==================================

收到收到～小雪你好呀！我这就把你的信息记小本本上，稍等一下下～
Tool Calls:
  save_user_info (call_00_p1pHE2fZIKYFUzAJqnYS2073)
 Call ID: call_00_p1pHE2fZIKYFUzAJqnYS2073
  Args:
    name: 小雪
    job: Java后端开发工程师
================================= Tool Message =================================
Name: save_user_info

saved user
================================== Ai Message ==================================

搞定啦～已经把你的信息稳稳存好了！

小雪你好呀，Java后端开发工程师，听起来就是那种天天跟 Spring、MySQL、Redis 斗智斗勇的狠人（笑）。以后有问题随时喊我，写代码卡壳了、想吐槽产品改需求了，都可以来找我聊～

顺便问一句，你平时是写微服务那一路的，还是偏业务 CRUD 呀？我好奇一下下～
================================ Human Message =================================

你还记得我吗？
================================== Ai Message ==================================

诶嘿～稍等哈，让我翻翻小本本看看有没有你的记录～
Tool Calls:
  get_user_info (call_00_G737qeraeX5q2Oj0Bwuo3421)
 Call ID: call_00_G73

#### 小结

1. 长期记忆用 Store 存储，结构是 `namespace + key -> value`；
2. `namespace` 是层级路径（元组），实现按用户/类别隔离；
3. `search` 按 namespace 前缀匹配，可用 `filter` 按 value 字段精确过滤；
4. `put` 相同 `namespace + key` 即更新；`delete` 删除单条；`list_namespaces` 列举层级；
5. `InMemoryStore` 与 `PostgresStore` 用法一致，后者可持久化、可跨进程共享；
6. 配置嵌入索引（`index={...}`）后，`search(query=...)` 可按语义检索（Postgres 需 pgvector）。